# Оценка моделей

## Импорты

In [1]:
from pathlib import Path
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from bert_score import score

C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Настройки

In [2]:
def find_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data").exists() and (folder / "reports").exists():
            return folder
    raise FileNotFoundError("Корень проекта не найден")


def style_plot(fig, title, subtitle, height=500):
    fig.update_layout(
        title={"text": f"{title}<br><sup>{subtitle}</sup>", "x": 0.02, "xanchor": "left"},
        template="plotly_white",
        height=height,
        font={"family": "Segoe UI", "size": 13, "color": "#1f2937"},
        margin={"l": 30, "r": 30, "t": 90, "b": 40},
        legend_title_text="",
    )
    fig.update_xaxes(gridcolor="#e5e7eb", zerolinecolor="#6b7280")
    fig.update_yaxes(gridcolor="#e5e7eb")
    return fig


ROOT = find_root()
REPORT_DIR = ROOT / "reports" / "evaluation"
RESULT_DIR = REPORT_DIR / "all_models"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
BERT_BATCH_SIZE = 16
BERT_DEVICE = None
BLUE = "#2563eb"
ORANGE = "#f59e0b"
PINK = "#db2777"
LIGHT_BLUE = "#93c5fd"

## GQA-ru

In [3]:
gqa_files = [
    ("Qwen2.5-VL базовая", "qwen_lora_v3_predictions.csv", "baseline_answer"),
    ("Qwen2.5-VL + LoRA #1", "qwen_lora_predictions.csv", "lora_answer"),
    ("Qwen2.5-VL + LoRA #2", "qwen_lora_v2_predictions.csv", "lora_answer"),
    ("Qwen2.5-VL + LoRA #3", "qwen_lora_v3_predictions.csv", "lora_answer"),
    ("SmolVLM базовая", "smolvlm_lora_predictions.csv", "baseline_answer"),
    ("SmolVLM + LoRA", "smolvlm_lora_predictions.csv", "lora_answer"),
]

def normalize(text):
    text = str(text).lower().replace("ё", "е").strip()
    text = re.sub(r"[^a-zа-я0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    answers = {"да": "yes", "нет": "no", "yes": "yes", "no": "no"}
    return answers.get(text, text)

gqa_parts = []
for model_name, file_name, prediction_column in gqa_files:
    part = pd.read_csv(REPORT_DIR / file_name)
    part = part[["id", "question", "true_answer", prediction_column]].copy()
    part.columns = ["id", "question", "reference", "prediction"]
    part["model"] = model_name
    gqa_parts.append(part)

gqa = pd.concat(gqa_parts, ignore_index=True)
gqa["id"] = gqa["id"].astype(str)
gqa["reference"] = gqa["reference"].fillna("").astype(str)
gqa["prediction"] = gqa["prediction"].fillna("").astype(str)
gqa["exact_match"] = gqa.apply(lambda row: normalize(row["reference"]) == normalize(row["prediction"]), axis=1)

gqa_data = ROOT / "data" / "GQA-ru" / "testdev_balanced_instructions" / "testdev-00000-of-00001.parquet"
gqa_types = pd.read_parquet(gqa_data, columns=["id", "types"])
gqa_types["id"] = gqa_types["id"].astype(str)
gqa_types["category"] = gqa_types["types"].apply(lambda value: value.get("semantic", "другое"))
gqa = gqa.merge(gqa_types[["id", "category"]], on="id", how="left")
gqa["category"] = gqa["category"].fillna("другое")
gqa.head()

,id,question,reference,prediction,model,exact_match,category
0,20412523,Человек сверху или снизу?,сверху,сверху,Qwen2.5-VL базовая,True,attr
1,201576820,"Вы видите и овец, и коз?",да,no,Qwen2.5-VL базовая,False,obj
2,20984271,Какого цвета фургон?,черный,черный,Qwen2.5-VL базовая,True,attr
3,201303485,Какой длины стол?,длинный,short,Qwen2.5-VL базовая,False,attr
4,20151632,Из чего сделаны очки?,пластик,plastic,Qwen2.5-VL базовая,False,attr


## Exact Match

In [4]:
gqa_exact = (
    gqa.groupby("model", as_index=False)
    .agg(exact_match=("exact_match", "mean"), examples=("exact_match", "size"))
    .sort_values("exact_match", ascending=False)
)
gqa_exact

,model,exact_match,examples
1,Qwen2.5-VL + LoRA #2,0.565,1000
2,Qwen2.5-VL + LoRA #3,0.551,1000
0,Qwen2.5-VL + LoRA #1,0.513,1000
3,Qwen2.5-VL базовая,0.321,1000
4,SmolVLM + LoRA,0.295,1000
5,SmolVLM базовая,0.212,1000


In [5]:
plot_data = gqa_exact.sort_values("exact_match")
fig = px.bar(plot_data, x="exact_match", y="model", orientation="h", text=plot_data["exact_match"].map(lambda x: f"{x:.1%}"), color_discrete_sequence=[BLUE])
fig.update_traces(textposition="outside", marker_line_color="#1e3a8a", marker_line_width=0.8)
fig.update_xaxes(range=[0, 0.65], tickformat=".0%", title="Exact Match")
fig.update_yaxes(title="")
style_plot(fig, "GQA-ru: Exact Match", "По 1000 примеров для каждой модели").show()

### Вывод по Exact Match

- Лучший результат теперь у `Qwen2.5-VL + LoRA #2` — **56,5%**.
- `Qwen2.5-VL + LoRA #3` получила **55,1%**, а `Qwen2.5-VL + LoRA #1` — **51,3%**.
- После обновления результатов первая версия больше не делит первое место со второй: разница между ними составляет **5,2 процентного пункта**.
- Базовая Qwen набрала **32,1%**. Прирост LoRA #1 относительно базы равен **19,2 п.п.**, LoRA #2 — **24,4 п.п.**, LoRA #3 — **23,0 п.п.**
- SmolVLM после LoRA выросла с **21,2%** до **29,5%**, но всё равно уступила всем версиям Qwen.

## BERTScore

In [6]:
bert_precision, bert_recall, bert_f1 = score(
    gqa["prediction"].tolist(),
    gqa["reference"].tolist(),
    lang="ru",
    batch_size=BERT_BATCH_SIZE,
    device=BERT_DEVICE,
    verbose=True,
)
gqa["bertscore_precision"] = bert_precision.cpu().numpy()
gqa["bertscore_recall"] = bert_recall.cpu().numpy()
gqa["bertscore_f1"] = bert_f1.cpu().numpy()

gqa_bert = (
    gqa.groupby("model", as_index=False)
    .agg(
        bertscore_precision=("bertscore_precision", "mean"),
        bertscore_recall=("bertscore_recall", "mean"),
        bertscore_f1=("bertscore_f1", "mean"),
    )
    .sort_values("bertscore_f1", ascending=False)
)
gqa_bert


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12434.32it/s]


[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.



  0%|          | 0/85 [00:00<?, ?it/s]

...


 90%|█████████ | 339/375 [00:01<00:00, 328.88it/s]


 99%|█████████▉| 373/375 [00:01<00:00, 327.14it/s]


100%|██████████| 375/375 [00:01<00:00, 324.11it/s]

done in 1.99 seconds, 3016.83 sentences/sec


,model,bertscore_precision,bertscore_recall,bertscore_f1
1,Qwen2.5-VL + LoRA #2,0.913522,0.908683,0.910873
2,Qwen2.5-VL + LoRA #3,0.911348,0.906437,0.908691
0,Qwen2.5-VL + LoRA #1,0.902359,0.898425,0.900177
4,SmolVLM + LoRA,0.854160,0.848785,0.851242
3,Qwen2.5-VL базовая,0.812672,0.804214,0.808191
5,SmolVLM базовая,0.691344,0.670577,0.680506


In [7]:
bert_plot = gqa_bert.sort_values("bertscore_f1")
fig = go.Figure()
fig.add_bar(name="Precision", y=bert_plot["model"], x=bert_plot["bertscore_precision"], orientation="h", marker_color=LIGHT_BLUE)
fig.add_bar(name="Recall", y=bert_plot["model"], x=bert_plot["bertscore_recall"], orientation="h", marker_color=ORANGE)
fig.add_bar(name="F1", y=bert_plot["model"], x=bert_plot["bertscore_f1"], orientation="h", marker_color=BLUE)
fig.update_layout(barmode="group")
fig.update_xaxes(range=[0.6, 0.95], tickformat=".0%", title="BERTScore")
fig.update_yaxes(title="")
style_plot(fig, "GQA-ru: BERTScore", "Семантическая близость ответов", 560).show()

### Вывод по BERTScore

- Лучший BERTScore F1 у `Qwen2.5-VL + LoRA #2` — **0,911**.
- LoRA #3 получила **0,909**, а LoRA #1 — **0,900**.
- У базовой Qwen F1 равен **0,808**, поэтому даже первая версия адаптера заметно улучшает смысловую близость ответов.
- SmolVLM с LoRA получила **0,851** против **0,681** у базовой модели.
- BERTScore заметно выше Exact Match: часть ответов близка по смыслу, но отличается формой слова, числом или формулировкой.

## Accuracy по категориям GQA-ru

In [8]:
gqa_categories = (
    gqa.groupby(["model", "category"], as_index=False)
    .agg(accuracy=("exact_match", "mean"), examples=("exact_match", "size"))
    .sort_values(["model", "accuracy"])
)
gqa_categories

,model,category,accuracy,examples
2,Qwen2.5-VL + LoRA #1,global,0.333333,6
4,Qwen2.5-VL + LoRA #1,rel,0.383178,428
1,Qwen2.5-VL + LoRA #1,cat,0.450000,80
0,Qwen2.5-VL + LoRA #1,attr,0.636150,426
3,Qwen2.5-VL + LoRA #1,obj,0.666667,60
7,Qwen2.5-VL + LoRA #2,global,0.333333,6
9,Qwen2.5-VL + LoRA #2,rel,0.443925,428
6,Qwen2.5-VL + LoRA #2,cat,0.525000,80
5,Qwen2.5-VL + LoRA #2,attr,0.673709,426
8,Qwen2.5-VL + LoRA #2,obj,0.733333,60


In [9]:
matrix = gqa_categories.pivot(index="model", columns="category", values="accuracy")
fig = px.imshow(matrix, text_auto=".1%", aspect="auto", zmin=0, zmax=1, color_continuous_scale=[[0, "#eff6ff"], [0.5, "#60a5fa"], [1, "#1e3a8a"]], labels={"color": "Accuracy"})
fig.update_xaxes(title="Категория")
fig.update_yaxes(title="")
style_plot(fig, "GQA-ru: качество по категориям", "Exact Match; global содержит только 6 примеров", 600).show()

### Вывод по категориям GQA-ru

- У Qwen LoRA #2 лучшие результаты среди крупных категорий получены для объектов — **73,3%** и атрибутов — **67,4%**.
- LoRA #1 получила **66,7%** на объектах, **63,6%** на атрибутах, **45,0%** на категориях объектов и **38,3%** на отношениях.
- LoRA #1 стала слабее LoRA #2 во всех крупных категориях. Самый большой разрыв виден в отношениях между объектами: **38,3%** против **44,4%**.
- Категория `global` содержит только 6 примеров, поэтому её результат нельзя считать устойчивым и использовать для выбора модели.
- У SmolVLM LoRA улучшила атрибуты и отношения, но итоговое качество всё ещё заметно ниже Qwen.

## MMBench-ru

In [10]:
mmbench = pd.read_csv(REPORT_DIR / "mmbench_ru_predictions.csv")
mmbench = mmbench[mmbench["error"].isna()].copy()
mmbench["is_correct"] = mmbench["is_correct"].astype(bool)
mmbench_metrics = (
    mmbench.groupby("model_name", as_index=False)
    .agg(accuracy=("is_correct", "mean"), examples=("is_correct", "size"))
    .sort_values("accuracy", ascending=False)
)
display(mmbench.head())
display(mmbench_metrics)

,model_name,family,row_id,parquet_row_id,split,category,task_type,metric_name,question,hint,options,true_answer,true_answer_norm,prediction,prediction_norm,is_correct,inference_time_s,error
0,Qwen2.5-VL базовая,qwen,110,8,dev,image_quality,multiple_choice,accuracy,Какая картина более яркая?,NaN,A. Первая картина\nB. Вторая картина,A,A,A,A,True,0.999790,NaN
1,Qwen2.5-VL базовая,qwen,321,12,dev,image_quality,multiple_choice,accuracy,Какая картинка более яркая?,NaN,A. Первая картинка\nB. Вторая картинка,A,A,A,A,True,0.234891,NaN
2,Qwen2.5-VL базовая,qwen,17,14,dev,image_quality,multiple_choice,accuracy,Какая из изображений более яркое?,NaN,A. Первое изображение\nB. Второе изображение,A,A,A,A,True,0.240894,NaN
3,Qwen2.5-VL базовая,qwen,179,17,dev,image_quality,multiple_choice,accuracy,Какое изображение более яркое?,NaN,A. Первое изображение\nB. Второе изображение,B,B,B,B,True,0.248059,NaN
4,Qwen2.5-VL базовая,qwen,237,26,dev,image_scene,multiple_choice,accuracy,Это место переполнено?,NaN,A. да\nB. нет,B,B,B,B,True,0.260959,NaN


,model_name,accuracy,examples
2,Qwen2.5-VL базовая,0.796,500
1,Qwen2.5-VL + LoRA #2,0.788,500
0,Qwen2.5-VL + LoRA #1,0.786,500
4,SmolVLM базовая,0.364,500
3,SmolVLM + LoRA,0.346,500


In [11]:
plot_data = mmbench_metrics.sort_values("accuracy")
fig = px.bar(plot_data, x="accuracy", y="model_name", orientation="h", text=plot_data["accuracy"].map(lambda x: f"{x:.1%}"), color_discrete_sequence=[ORANGE])
fig.update_traces(textposition="outside", marker_line_color="#92400e", marker_line_width=0.8)
fig.update_xaxes(range=[0, 0.88], tickformat=".0%", title="Accuracy")
fig.update_yaxes(title="")
style_plot(fig, "MMBench-ru: Accuracy", "По 500 примеров для каждой модели").show()

### Вывод по MMBench-ru

- Базовая Qwen показала лучший результат — **79,6%**.
- Qwen LoRA #1 и #2 получили **78,6%** и **78,8%**: улучшения относительно базовой модели нет.
- Базовая SmolVLM набрала **36,4%**, версия с LoRA — **34,6%**.
- LoRA хорошо повысила качество на целевой GQA-ru, но не улучшила перенос на более широкий MMBench-ru.

## Accuracy по категориям MMBench-ru

In [12]:
mmbench_categories = (
    mmbench.groupby(["model_name", "category"], as_index=False)
    .agg(accuracy=("is_correct", "mean"), examples=("is_correct", "size"))
    .sort_values(["model_name", "accuracy"])
)
mmbench_categories

,model_name,category,accuracy,examples
13,Qwen2.5-VL + LoRA #1,object_localization,0.395349,43
18,Qwen2.5-VL + LoRA #1,spatial_relationship,0.400000,20
8,Qwen2.5-VL + LoRA #1,image_quality,0.448276,29
16,Qwen2.5-VL + LoRA #1,physical_relation,0.500000,10
15,Qwen2.5-VL + LoRA #1,physical_property_reasoning,0.578947,19
...,...,...,...,...
88,SmolVLM базовая,image_quality,0.448276,29
91,SmolVLM базовая,image_topic,0.466667,15
89,SmolVLM базовая,image_scene,0.473684,57
86,SmolVLM базовая,identity_reasoning,0.500000,22


In [13]:
matrix = mmbench_categories.pivot(index="model_name", columns="category", values="accuracy")
fig = px.imshow(matrix, text_auto=".0%", aspect="auto", zmin=0, zmax=1, color_continuous_scale=[[0, "#fff7ed"], [0.5, "#fb923c"], [1, "#9a3412"]], labels={"color": "Accuracy"})
fig.update_xaxes(title="Категория", tickangle=-40)
fig.update_yaxes(title="")
style_plot(fig, "MMBench-ru: качество по категориям", "Accuracy для каждой модели и категории", 700).show()

In [14]:
category_total = (
    mmbench.groupby("category", as_index=False)
    .agg(accuracy=("is_correct", "mean"), examples=("is_correct", "size"))
    .sort_values("accuracy")
)
fig = px.bar(category_total, x="accuracy", y="category", orientation="h", text=category_total["accuracy"].map(lambda x: f"{x:.1%}"), color_discrete_sequence=[PINK], hover_data=["examples"])
fig.update_traces(textposition="outside", marker_line_color="#831843", marker_line_width=0.6)
fig.update_xaxes(range=[0, 0.9], tickformat=".0%", title="Средняя Accuracy")
fig.update_yaxes(title="")
style_plot(fig, "MMBench-ru: сложность категорий", "Среднее по всем моделям", 720).show()

### Вывод по категориям MMBench-ru

- Самые сильные категории: `identity_reasoning` — **80,0%**, `image_scene` — **77,5%**, `image_emotion` — **74,2%**.
- Самые сложные: `object_localization` — **34,4%** и `spatial_relationship` — **36,0%**.
- Qwen опережает SmolVLM почти во всех категориях. Основная зона улучшения — локализация и пространственные отношения.

## Эффект LoRA

In [15]:
lora_effect = pd.DataFrame([
    ["GQA-ru", "Qwen LoRA #1", 0.513 - 0.321], ["GQA-ru", "Qwen LoRA #2", 0.565 - 0.321],
    ["GQA-ru", "Qwen LoRA #3", 0.551 - 0.321], ["GQA-ru", "SmolVLM LoRA", 0.295 - 0.212],
    ["MMBench-ru", "Qwen LoRA #1", 0.786 - 0.796], ["MMBench-ru", "Qwen LoRA #2", 0.788 - 0.796],
    ["MMBench-ru", "SmolVLM LoRA", 0.346 - 0.364],
], columns=["dataset", "model", "change"])
display(lora_effect)
fig = px.bar(lora_effect, x="model", y="change", color="dataset", barmode="group", text=lora_effect["change"].map(lambda x: f"{x:+.1%}"), color_discrete_map={"GQA-ru": BLUE, "MMBench-ru": ORANGE})
fig.update_traces(textposition="outside")
fig.add_hline(y=0, line_color="#374151")
fig.update_yaxes(tickformat="+.0%", title="Изменение относительно базовой модели")
fig.update_xaxes(title="")
style_plot(fig, "Эффект LoRA", "Изменение основной метрики относительно базовой модели", 560).show()

,dataset,model,change
0,GQA-ru,Qwen LoRA #1,0.192
1,GQA-ru,Qwen LoRA #2,0.244
2,GQA-ru,Qwen LoRA #3,0.230
3,GQA-ru,SmolVLM LoRA,0.083
4,MMBench-ru,Qwen LoRA #1,-0.010
5,MMBench-ru,Qwen LoRA #2,-0.008
6,MMBench-ru,SmolVLM LoRA,-0.018


### Вывод по эффекту LoRA

Все адаптеры улучшили качество на целевой GQA-ru, но величина прироста различается. Qwen LoRA #1 даёт **+19,2 п.п.**, Qwen LoRA #2 — **+24,4 п.п.**, Qwen LoRA #3 — **+23,0 п.п.**, а SmolVLM LoRA — **+8,3 п.п.** На MMBench-ru изменения остаются небольшими и отрицательными: **−0,8…−1,8 п.п.** Это подтверждает специализацию моделей на GQA-ru и небольшую потерю универсальности.

## Итоги

In [16]:
gqa_results = gqa_exact.merge(gqa_bert, on="model")
gqa_results["dataset"] = "GQA-ru"
mmbench_results = mmbench_metrics.rename(columns={"model_name": "model"})
mmbench_results["dataset"] = "MMBench-ru"

gqa_results.to_csv(RESULT_DIR / "gqa_metrics.csv", index=False, encoding="utf-8-sig")
gqa_categories.to_csv(RESULT_DIR / "gqa_category_accuracy.csv", index=False, encoding="utf-8-sig")
mmbench_results.to_csv(RESULT_DIR / "mmbench_metrics.csv", index=False, encoding="utf-8-sig")
mmbench_categories.to_csv(RESULT_DIR / "mmbench_category_accuracy.csv", index=False, encoding="utf-8-sig")
gqa.to_csv(RESULT_DIR / "gqa_scored_predictions.csv", index=False, encoding="utf-8-sig")
mmbench.to_csv(RESULT_DIR / "mmbench_scored_predictions.csv", index=False, encoding="utf-8-sig")

display(gqa_results.sort_values("exact_match", ascending=False))
display(mmbench_results)

,model,exact_match,examples,bertscore_precision,bertscore_recall,bertscore_f1,dataset
0,Qwen2.5-VL + LoRA #2,0.565,1000,0.913522,0.908683,0.910873,GQA-ru
1,Qwen2.5-VL + LoRA #3,0.551,1000,0.911348,0.906437,0.908691,GQA-ru
2,Qwen2.5-VL + LoRA #1,0.513,1000,0.902359,0.898425,0.900177,GQA-ru
3,Qwen2.5-VL базовая,0.321,1000,0.812672,0.804214,0.808191,GQA-ru
4,SmolVLM + LoRA,0.295,1000,0.854160,0.848785,0.851242,GQA-ru
5,SmolVLM базовая,0.212,1000,0.691344,0.670577,0.680506,GQA-ru


,model,accuracy,examples,dataset
2,Qwen2.5-VL базовая,0.796,500,MMBench-ru
1,Qwen2.5-VL + LoRA #2,0.788,500,MMBench-ru
0,Qwen2.5-VL + LoRA #1,0.786,500,MMBench-ru
4,SmolVLM базовая,0.364,500,MMBench-ru
3,SmolVLM + LoRA,0.346,500,MMBench-ru


### Общий вывод

По обновлённым результатам лучшей моделью для основной задачи проекта является **`Qwen2.5-VL + LoRA #2`**. Она получила **56,5% Exact Match** и **0,911 BERTScore F1** на 1000 примерах GQA-ru. Эта версия лидирует и по строгому совпадению ответов, и по их семантической близости.

**Qwen LoRA #1** занимает третье место среди адаптеров Qwen: **51,3% Exact Match** и **0,900 BERTScore F1**. Она всё ещё значительно лучше базовой Qwen с 32,1%, но уступает LoRA #2 на 5,2 процентного пункта и LoRA #3 на 3,8 процентного пункта. Следовательно, прежний вывод о равенстве первой и второй версий больше не поддерживается данными.

**Qwen LoRA #3** показала 55,1% Exact Match и 0,909 BERTScore F1. Она близка к LoRA #2, но дополнительное обучение не привело к новому максимуму. Это может означать, что после определённого числа шагов качество вышло на плато или начало немного снижаться.

Разбор категорий также поддерживает выбор LoRA #2. Она получила **73,3%** на вопросах об объектах, **67,4%** на атрибутах и **44,4%** на отношениях. У обновлённой LoRA #1 соответствующие результаты ниже: **66,7%**, **63,6%** и **38,3%**. Категория `global` слишком мала — всего 6 примеров — поэтому по ней нельзя делать надёжные выводы.

LoRA хорошо решает именно задачу, на которой проводилось обучение. Для Qwen прирост на GQA-ru составляет от **+19,2 до +24,4 п.п.**, а для SmolVLM — **+8,3 п.п.** При этом на MMBench-ru лучшей остаётся базовая Qwen с **79,6% Accuracy**. LoRA #1 получила 78,6%, LoRA #2 — 78,8%, поэтому явного улучшения универсального качества нет.

Qwen заметно сильнее SmolVLM во всех основных сравнениях. SmolVLM LoRA улучшила GQA-ru с 21,2% до 29,5%, но этого недостаточно, чтобы приблизиться к результатам Qwen. На MMBench-ru разрыв ещё больше: 34,6% у SmolVLM LoRA против 78,8% у Qwen LoRA #2.

Самыми сложными направлениями остаются пространственные отношения и локализация объектов. На MMBench-ru средняя Accuracy для `spatial_relationship` равна **36,0%**, а для `object_localization` — **34,4%**. При следующем обучении стоит добавить больше примеров на положение объектов, направления, расстояния и привязку ответа к конкретной области изображения.